In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath(".."))

import pandas as pd

from simulation import simulate_many_from_snapshot

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"

PROCESSED_PATH = DATA_DIR / "processed"
LIVE_PATH = DATA_DIR / "live"

### live_results.csv

raw input, one row per contestant per episode

- contestant
- episode
- episode_score

✅ Done

In [7]:
live_results = pd.read_csv(LIVE_PATH / "live_results.csv")

live_results

,contestant,episode,episode_score
0,Amy Gledhill,1,18
1,Armando Iannucci,1,16
2,Joanna Page,1,12
3,Joel Dommett,1,9
4,Kumail Nanjiani,1,13
5,Amy Gledhill,2,12
6,Armando Iannucci,2,20
7,Joanna Page,2,8
8,Joel Dommett,2,15
9,Kumail Nanjiani,2,10


### latest_snapshot.csv

One row per contestant

MVP:
- contestant

- latest_episode_score
- cumulative_score
- current_rank
- win_probability

✅ Done


Later:
- latest_episode
- predicted_final_score
- archetype
- archetype_description

In [9]:
# ----- Build live_snapshot (temporary) -----

live_snapshot = (
    live_results
    .sort_values(["contestant", "episode"])
    .groupby("contestant")["episode_score"]
    .agg(["sum", "mean", "std", "count"])
    .reset_index()
    .rename(columns={
        "sum": "cumulative_score",
        "mean": "mean_score_so_far",
        "std": "std_score_so_far",
        "count": "episodes_played"
    })
)

total_episodes = 10

live_snapshot["remaining_episodes"] = (
    total_episodes - live_snapshot["episodes_played"]
)

# fix std
global_std = live_results["episode_score"].std()

live_snapshot["std_score_so_far"] = live_snapshot["std_score_so_far"].fillna(global_std)

live_snapshot.loc[
    live_snapshot["std_score_so_far"] == 0,
    "std_score_so_far"
] = global_std


# ----- Run simulations -----

live_simulations = simulate_many_from_snapshot(
    live_snapshot,
    n_simulations=1000
)


# ----- Convert to win probabilities -----

win_probs = (
    live_simulations[live_simulations["rank"] == 1]
    .groupby("contestant")
    .size()
    .reset_index(name="wins")
)

win_probs["win_probability"] = win_probs["wins"] / 1000


# ----- Build latest_snapshot -----

# cumulative score
cum_scores = (
    live_results
    .groupby("contestant")["episode_score"]
    .sum()
    .reset_index(name="cumulative_score")
)

# latest episode score
latest_ep = live_results["episode"].max()

latest_scores = (
    live_results[live_results["episode"] == latest_ep]
    [["contestant", "episode_score"]]
    .rename(columns={"episode_score": "latest_episode_score"})
)

# merge
latest_snapshot = cum_scores.merge(win_probs, on="contestant", how="left")
latest_snapshot = latest_snapshot.merge(latest_scores, on="contestant", how="left")

# rank
latest_snapshot["current_rank"] = (
    latest_snapshot["cumulative_score"]
    .rank(ascending=False, method="min")
)

latest_snapshot = latest_snapshot.fillna(0)

# sort nicely
latest_snapshot = latest_snapshot.sort_values(
    "win_probability", ascending=False
).reset_index(drop=True)


# ----- Select only the required columns -----

latest_snapshot = latest_snapshot[
    [
        "contestant",
        "latest_episode_score",
        "cumulative_score",
        "current_rank",
        "win_probability"
    ]
]

# clean types
latest_snapshot["current_rank"] = latest_snapshot["current_rank"].astype(int)


# ----- Save snapshot -----

latest_snapshot.to_csv(PROCESSED_PATH / "latest_snapshot.csv", index=False)

### win_probability_trajectory.csv

One row per contestant per episode:

- contestant
- episode
- win_probability

✅ Done

In [ ]:
# ----- Recalculate win probabilities at each episode from 1 to latest -----
# ----- Save snapshots for each episode, calculate trajectory of win probabilities -----

total_episodes = 10
n_simulations = 1000

def build_live_snapshot(results_df, total_episodes, global_std):

    live_snapshot = (
        results_df
        .sort_values(["contestant", "episode"])
        .groupby("contestant")["episode_score"]
        .agg(["sum", "mean", "std", "count"])
        .reset_index()
        .rename(columns={
            "sum": "cumulative_score",
            "mean": "mean_score_so_far",
            "std": "std_score_so_far",
            "count": "episodes_played"
        })
    )

    live_snapshot["remaining_episodes"] = (
        total_episodes - live_snapshot["episodes_played"]
    )

    # fix std
    live_snapshot["std_score_so_far"] = live_snapshot["std_score_so_far"].fillna(global_std)

    live_snapshot.loc[
        live_snapshot["std_score_so_far"] == 0,
        "std_score_so_far"
    ] = global_std

    return live_snapshot

# Global fallback std from historical data if available, otherwise from live results
global_std = live_results["episode_score"].std()

trajectory = []

latest_episode = live_results["episode"].max()

for episode_t in range(1, latest_episode + 1):

    # Keep only results up to episode_t
    results_so_far = live_results[live_results["episode"] <= episode_t].copy()

    # Build modelling snapshot for episode_t
    live_snapshot_t = build_live_snapshot(results_so_far, total_episodes, global_std)

    # Run simulations for episode_t
    live_simulations_t = simulate_many_from_snapshot(
        live_snapshot_t,
        n_simulations=n_simulations
    )

    # Convert to win probabilities for episode_t
    win_probs_t = (
        live_simulations_t[live_simulations_t["rank"] == 1]
        .groupby("contestant")
        .size()
        .reset_index(name="wins")
    )

    win_probs_t["win_probability"] = win_probs_t["wins"] / n_simulations
    win_probs_t["episode"] = episode_t

    # Include all contestants, even with 0 wins
    all_contestants = pd.DataFrame({
        "contestant": live_snapshot_t["contestant"]
    })

    win_probs_t = all_contestants.merge(
        win_probs_t[["contestant", "win_probability", "episode"]],
        on="contestant",
        how="left"
    )

    # Fill NaN win probabilities with 0 for contestants who didn't win any simulations
    win_probs_t["win_probability"] = win_probs_t["win_probability"].fillna(0)
    win_probs_t["episode"] = win_probs_t["episode"].fillna(episode_t)

    win_probs_t["episode"] = win_probs_t["episode"].astype(int)

    trajectory.append(win_probs_t)

# Combine trajectory data from all episodes
trajectory_df = pd.concat(trajectory, ignore_index=True)

# Reorder and save
trajectory_df = trajectory_df[["contestant", "episode", "win_probability"]]

trajectory_df.to_csv(PROCESSED_PATH / "win_probability_trajectory.csv", index=False)

trajectory_df.head(20)

,contestant,episode,win_probability
0,Amy Gledhill,1,0.872
1,Armando Iannucci,1,0.128
2,Joanna Page,1,0.000
3,Joel Dommett,1,0.000
4,Kumail Nanjiani,1,0.000
5,Amy Gledhill,2,0.016
6,Armando Iannucci,2,0.984
7,Joanna Page,2,0.000
8,Joel Dommett,2,0.000
9,Kumail Nanjiani,2,0.000


### score_trajectory.csv

One row per contestant per episode:

- contestant
- episode
- episode_score
- cumulative_score

✅ Done

In [19]:
score_trajectory = live_results.copy()

score_trajectory = score_trajectory.sort_values(["contestant", "episode"])

score_trajectory["cumulative_score"] = (
    score_trajectory
    .groupby("contestant")["episode_score"]
    .cumsum()
)

score_trajectory.to_csv(PROCESSED_PATH / "score_trajectory.csv", index=False)

score_trajectory.head(20)

,contestant,episode,episode_score,cumulative_score
0,Amy Gledhill,1,18,18
5,Amy Gledhill,2,12,30
1,Armando Iannucci,1,16,16
6,Armando Iannucci,2,20,36
2,Joanna Page,1,12,12
7,Joanna Page,2,8,20
3,Joel Dommett,1,9,9
8,Joel Dommett,2,15,24
4,Kumail Nanjiani,1,13,13
9,Kumail Nanjiani,2,10,23


### archetype_examples.csv

- archetype
- archetype_description
- example_contestant

### archetypes_current.csv

### latest_note.txt or json